In [ ]:
import warnings
warnings.filterwarnings('ignore')

import matplotlib.pyplot as plt

from unseen import eva
from unseen import similarity
from unseen import stability

import utils

In [ ]:
# optional parameters
# (this cell is tagged "parameters")

#metric = 'txx'
#location = 'Miena'
#location [0, 11]

In [ ]:
assert "location" in locals(), "Must provide a location name"
assert "metric" in locals(), "Must provide a metric (rx1day or txx)"

In [ ]:
var_dict = {'txx': 'tasmax', 'rx1day': 'pr'}
var = var_dict[metric]

### Observations

In [ ]:
da_obs = utils.get_obs_data(metric, location)

In [ ]:
df_obs = da_obs.to_dataframe()

In [ ]:
ranked_years = df_obs.sort_values(by=var, ascending=False)
print(ranked_years.head(n=10))

In [ ]:
da_obs_detrended, linear_data_obs = utils.detrend_obs(da_obs)

In [ ]:
gev_obs_detrended = list(eva.fit_gev(da_obs_detrended.values))

In [ ]:
fig, ax = plt.subplots(figsize=[9, 5])
years = da_obs['time'].dt.year.values
plt.plot(years, da_obs.values, marker='o', label='raw data', color='tab:blue', alpha=0.5)
plt.plot(years, linear_data_obs, color='tab:cyan')
plt.plot(years, da_obs_detrended.values, marker='o', label='detrended data', color='tab:orange', alpha=0.5)
plt.xlim(years[0] - 0.5, years[-1] + 0.5)
plt.title(metric)
plt.ylabel(metric)
plt.xlabel('year')
plt.legend()
plt.grid()

### Model data

In [ ]:
def process_model(model):
    """Process one model."""
    
    # Raw model data
    da_model_stacked = utils.get_model_data(metric, model, location)
    da_model_detrended, da_model_detrended_stacked, linear_data_model = utils.detrend_model(da_model_stacked)
    gev_model_detrended = list(eva.fit_gev(da_model_detrended_stacked.values))
    utils.plot_model_data(da_model_stacked, da_model_detrended_stacked, linear_data_model, metric)
    stability.create_plot(
        da_model_stacked.unstack(),
        metric,
        [1960, 1970, 1980, 1990, 2000, 2010],
        uncertainty=True,
        return_method='gev',
        units=metric,
#        ylim=(0, 320),
    )
    stability.create_plot(
        da_model_detrended.unstack(),
        f'{metric} (detrended)',
        [1960, 1970, 1980, 1990, 2000, 2010],
        uncertainty=True,
        return_method='gev',
        units=metric,
#        ylim=(0, 320),
    )
    # Mean correction
    da_model_detrended_bc_mean_stacked = utils.mean_correction(da_model_detrended, da_obs_detrended, metric)
    gev_model_detrended_bc_mean = list(eva.fit_gev(da_model_detrended_bc_mean_stacked.values))
    # Quantile correction
    da_model_detrended_bc_quantile_stacked = utils.quantile_correction(
        da_model_detrended_stacked,
        da_obs_detrended,
        metric,
        plot_af=True
    )
    gev_model_detrended_bc_quantile = list(eva.fit_gev(da_model_detrended_bc_quantile_stacked.values))
    # Plot distributions and return periods
    utils.plot_distributions(
        metric,
        location,
        da_obs_detrended,
        gev_obs_detrended,
        da_model_detrended_stacked,
        gev_model_detrended,
        da_model_detrended_bc_mean_stacked,
        gev_model_detrended_bc_mean,
        da_model_detrended_bc_quantile_stacked,
        gev_model_detrended_bc_quantile,
    )
    utils.plot_return_curves(
        metric,
        location,
        da_obs_detrended,
        gev_obs_detrended,
        da_model_detrended,
        gev_model_detrended,
        da_model_detrended_bc_mean_stacked.unstack(),
        gev_model_detrended_bc_mean,
        da_model_detrended_bc_quantile_stacked.unstack(),
        gev_model_detrended_bc_quantile,
    )
    # Similarity testing
    print('Raw data similarity tests')
    similarity_ds = similarity.similarity_tests(da_model_detrended, da_obs_detrended)
    print('KS score:', similarity_ds['ks_statistic'].values)
    print('KS p-value:', similarity_ds['ks_pval'].values)
    print('AD score:', similarity_ds['ad_statistic'].values)
    print('AD p-value:', similarity_ds['ad_pval'].values)
    print('Mean correction similarity tests')
    utils.fidelity_tests(da_model_detrended, da_obs_detrended, da_model_detrended_bc_mean_stacked.unstack())
    print('Quantile correction similarity tests')
    utils.fidelity_tests(da_model_detrended, da_obs_detrended, da_model_detrended_bc_quantile_stacked.unstack())

In [ ]:
process_model('BCC-CSM2-MR')

In [ ]:
process_model('CAFE')

In [ ]:
process_model('CMCC-CM2-SR5')

In [ ]:
process_model('CanESM5')

In [ ]:
process_model('EC-Earth3')

In [ ]:
process_model('IPSL-CM6A-LR')

In [ ]:
process_model('MIROC6')

In [ ]:
process_model('MPI-ESM1-2-HR')

In [ ]:
process_model('MRI-ESM2-0')

In [ ]:
process_model('NorCPM1')